# 02 — Labelling development

How rule-based labels become bounding boxes, and how often each pattern actually occurs.

**The thing to keep in mind throughout:** these labels are TA-Lib's opinion, not a human's.
This notebook is where that becomes concrete — you can look at a box and judge for yourself
whether you would have called it the same pattern.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # import src/ from notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config
from src.data_pipeline.fetch_ohlc import fetch_ohlc
from src.labeling.talib_labeler import label_patterns, pattern_counts

bars = fetch_ohlc(config.TICKER)
labels = label_patterns(bars)
print(f'{len(labels)} pattern instances over {len(bars)} bars')
labels.head(10)

## Class balance is severe, and it is not a bug

Doji fires an order of magnitude more often than the three-candle patterns. This is a
property of the rules, not of the sampling, and it is why detection results are reported
per class instead of as one averaged mAP.

In [ ]:
counts = pattern_counts(labels)
fig, ax = plt.subplots(figsize=(9, 4))
colours = ['#2E7D32' if config.PATTERN_DIRECTION[c] > 0 else
           '#C62828' if config.PATTERN_DIRECTION[c] < 0 else '#757575'
           for c in counts.index]
ax.bar(counts.index, counts.values, color=colours)
ax.set_yscale('log'); ax.set_ylabel('instances (log)')
ax.set_title(f'{config.TICKER}: pattern frequency, whole history')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()
print(counts.to_string())

## Why SPY alone is not enough

In the training period the two three-candle classes fall below the minimum-instance floor.
That is the reason auxiliary tickers exist — dropping two of eight classes was the
alternative.

In [ ]:
train_hits = labels[labels.date <= config.TRAIN_END]
tc = train_hits.pattern.value_counts().reindex(config.CLASSES, fill_value=0)
for c in config.CLASSES:
    flag = '  <-- below the floor' if tc[c] < config.MIN_INSTANCES_PER_CLASS else ''
    print(f'{c:<20}{tc[c]:>6}{flag}')
print(f'\nfloor = {config.MIN_INSTANCES_PER_CLASS} instances')

## From a rule hit to a box on a picture

A multi-bar pattern is reported by TA-Lib on the bar where it *completes*, so a three-bar
Morning Star flagged at index i occupies bars i-2..i. The box spans all of them. Geometry
comes from the rendered figure's own transform, so it is exact rather than estimated.

In [ ]:
from src.data_pipeline.render_charts import render_window
from src.labeling.bbox_utils import pattern_bbox
from src.labeling.build_dataset import _labels_in_window

by_pos = {}
for rec in labels.to_dict('records'): by_pos.setdefault(rec['position'], []).append(rec)

def show(end_pos):
    """Render the window ending at end_pos with every in-frame pattern boxed."""
    start = end_pos - config.WINDOW + 1
    win = bars.iloc[start:end_pos + 1]
    img, mapper = render_window(win)
    fig, ax = plt.subplots(figsize=(7.5, 7.5)); ax.imshow(img); ax.axis('off')
    for hit in _labels_in_window(by_pos, end_pos, config.WINDOW):
        last_c = hit['position'] - start; first_c = last_c - hit['span'] + 1
        seg = bars.iloc[start + first_c: start + last_c + 1]
        cx, cy, w, h = pattern_bbox(mapper, float(seg['High'].max()),
                                    float(seg['Low'].min()), first_c, last_c)
        x, y = (cx - w/2) * img.shape[1], (cy - h/2) * img.shape[0]
        colour = ('#2E7D32' if config.PATTERN_DIRECTION[hit['pattern']] > 0 else
                  '#C62828' if config.PATTERN_DIRECTION[hit['pattern']] < 0 else '#455A64')
        ax.add_patch(plt.Rectangle((x, y), w*img.shape[1], h*img.shape[0],
                                   fill=False, edgecolor=colour, lw=2))
        ax.text(x, y - 4, hit['pattern'], color=colour, fontsize=8, weight='bold')
    ax.set_title(f"{config.TICKER} window ending {win.index[-1].date()}")
    plt.tight_layout(); plt.show()

show(len(bars) - 1)

## A rare three-candle pattern

Find a Morning Star and look at it. Does the rule's call match what you would have marked?
That question is the whole limitation of weak labels, in one picture.

In [ ]:
ms = labels[labels.pattern == 'MorningStar']
print(f'{len(ms)} Morning Stars; showing the most recent')
if len(ms):
    show(int(ms.position.iloc[-1]))

## Do patterns cluster in time?

If hits bunched into a few regimes, the chronological splits could hand one split most of
a class. Worth checking before trusting per-class test numbers.

In [ ]:
per_year = labels.groupby([labels.date.dt.year, 'pattern']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(13, 4.5))
per_year.plot(kind='area', stacked=True, ax=ax, lw=0, alpha=0.85, colormap='tab10')
ax.set_title('pattern hits per year'); ax.set_xlabel('year'); ax.grid(alpha=0.2)
ax.legend(fontsize=7, ncol=4); plt.tight_layout(); plt.show()